In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.model_selection import StratifiedKFold, train_test_split
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
#Mount Google Drive to access our dataset stored there instead of uploading files repeatedly
drive.mount('/content/drive')

In [ ]:
#Define the base directory where our Alzheimer's disease neuroimaging dataset is stored
BASE_DIR = '/content/drive/MyDrive/Final_Dataset1'

#Define folder structure based on our dataset organization:
# - Modality types: MRI, PET, co-registered (coreg)
# - Patient classes: Alzheimer's Disease (ad) vs Control Normal (cn)
# - Data augmentation status: original vs augmented

folders = [
    'original_mri_ad', 'original_mri_cn',
    'augmented_mri_ad', 'augmented_mri_cn',
    'original_pet_ad', 'original_pet_cn',
    'augmented_pet_ad', 'augmented_pet_cn',
    'coreg_ad_augmented', 'coreg_ad_original',
    'coreg_cn_augmented', 'coreg_cn_original'
]


In [ ]:
#Load neuroimaging data from NIfTI format which is the standard for brain imaging
def load_nifti_file(file_path):
    try:
        img = nib.load(file_path)
        #Convert to numpy array for analysis since we need numerical access to voxel values
        data = img.get_fdata()
        #Some scans might have time as 4th dimension; we only need spatial data for this analysis
        if len(data.shape) == 4:
            data = data[:, :, :, 0]
        return data, img.header
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

In [ ]:
#Build a catalogue of dataset entries to streamline loading, validation, and analysis without repeatedly scanning directories.
def datasetCatalogue(base_dir, folders):
    catalogue = []

    for folder in folders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
        #Skip folders that don't exist to allow flexible folder management and avoid runtime errors.
            continue

        is_augmented = 'augmented' in folder #Identify augmented data early to enable separate evaluation if needed.
        is_coregistered = 'coreg' in folder  #Identify co-registered scans because they have different preprocessing assumptions.

        if is_coregistered:
            modality = 'coreg' #Treat co-registered data as a distinct modality for better downstream handling.
            label = 1 if 'ad' in folder else 0 #Infer labels from folder naming convention to avoid manual annotation.
        else:
            modality = 'mri' if 'mri' in folder else 'pet' #Default to standard modalities unless otherwise indicated.
            label = 1 if 'ad' in folder else 0


        nifti_files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]

        for nifti_file in nifti_files:
            file_path = os.path.join(folder_path, nifti_file)

            try:
                img = nib.load(file_path)
                shape = img.shape
                spacing = img.header.get_zooms()
            except Exception as e:
              #Continue processing other files even if some files are corrupt or unreadable, ensuring maximum dataset is used
                print(f"Error looking at {file_path}: {e}")
                continue

            #Consolidate relevant metadata to allow flexible querying, stratification, and data quality analysis later.
            catalogue.append({'file_path': file_path, 'filename': nifti_file, 'folder': folder, 'modality': modality, 'label': label, 'is_augmented': is_augmented, 'is_coregistered': is_coregistered, 'shape': shape,'spacing': spacing})

    df = pd.DataFrame(catalogue)
    return df

catalogue_df = datasetCatalogue(BASE_DIR, folders)

#Provide quick dataset health checks to detect class imbalance, missing modalities, or preprocessing issues early.
print(f"Total files: {len(catalogue_df)}")
print(f"MRI files: {len(catalogue_df[catalogue_df['modality'] == 'mri'])}")
print(f"PET files: {len(catalogue_df[catalogue_df['modality'] == 'pet'])}")
print(f"Co-registered files: {len(catalogue_df[catalogue_df['modality'] == 'coreg'])}")
print(f"AD samples: {len(catalogue_df[catalogue_df['label'] == 1])}")
print(f"CN samples: {len(catalogue_df[catalogue_df['label'] == 0])}")
print(f"Original samples: {len(catalogue_df[catalogue_df['is_augmented'] == False])}")
print(f"Augmented samples: {len(catalogue_df[catalogue_df['is_augmented'] == True])}")


print("\nShape distribution:")
print(catalogue_df.groupby(['modality', 'shape']).size())

In [ ]:
#Split the dataset into training, validation, and test sets while preserving label balance and ensuring reproducibility across splits.
def datasetSplitting(catalogue_df, test_size=0.2, val_size=0.15, random_state=42):

    #Use empty DataFrames to allow collection of splits across modalities
    train_df = pd.DataFrame()
    val_df = pd.DataFrame()
    test_df = pd.DataFrame()

    #Process each modality separately to avoid introducing modality bias during train/val/test splitting.
    for modality in ['mri', 'pet', 'coreg']:

        modality_df = catalogue_df[catalogue_df['modality'] == modality]  #Isolate modality-specific samples to treat structural and functional data appropriately.
        original_df = modality_df[~modality_df['is_augmented']]            #Only split original (non-augmented) data to maintain fair and unbiased validation and test sets.
        augmented_df = modality_df[modality_df['is_augmented']]            #Augmented data is reserved for training to enrich learning without contaminating evaluation.

        #Initial split to separate out a fixed percentage of the data for unbiased testing.
        train_val_df, test_mod_df = train_test_split(original_df, test_size=test_size, random_state=random_state, stratify=original_df['label'])  #Maintain the class ratio across splits to prevent skewed evaluation.

        #Adjust validation size relative to the remaining train_val pool after test samples have been removed.
        adjusted_val_size = val_size / (1 - test_size)

        #Secondary split to carve out a validation set from the remaining data, again enforcing class balance.
        train_mod_df, val_mod_df = train_test_split(train_val_df, test_size=adjusted_val_size, random_state=random_state, stratify=train_val_df['label'])

        #Augmented samples are intentionally injected into the training set only to improve model generalisation without biasing validation or test results.
        train_mod_df = pd.concat([train_mod_df, augmented_df])

        #Merge splits from each modality to construct the complete training, validation, and test datasets.
        train_df = pd.concat([train_df, train_mod_df])
        val_df = pd.concat([val_df, val_mod_df])
        test_df = pd.concat([test_df, test_mod_df])

    return train_df, val_df, test_df

train_df, val_df, test_df = datasetSplitting(catalogue_df)

#Provide quick visibility into label balance to detect possible stratification or split errors early.
print(f"Training set: {len(train_df)} files ({len(train_df[train_df['label'] == 1])} AD, {len(train_df[train_df['label'] == 0])} CN)")
print(f"Validation set: {len(val_df)} files ({len(val_df[val_df['label'] == 1])} AD, {len(val_df[val_df['label'] == 0])} CN)")
print(f"Test set: {len(test_df)} files ({len(test_df[test_df['label'] == 1])} AD, {len(test_df[test_df['label'] == 0])} CN)")

#Check the distribution of modalities per split to verify no modality has been inadvertently excluded or overrepresented.
print("\nModality distribution in training set:")
print(train_df.groupby(['modality', 'label']).size())
print("\nModality distribution in validation set:")
print(val_df.groupby(['modality', 'label']).size())
print("\nModality distribution in test set:")
print(test_df.groupby(['modality', 'label']).size())


In [ ]:
#Create a custom dataset class to manage loading, preprocessing, and batching of NIfTI medical imaging data.
class NIfTIDataset(Dataset):
    def __init__(self, dataframe, modality=None, transform=None):

        self.transform = transform

        #Filter the dataset by modality if specified to allow flexibility in training modality-specific models.
        if modality:
            self.data = dataframe[dataframe['modality'] == modality].reset_index(drop=True)
        else:
            self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        #Return the total number of samples to integrate seamlessly with PyTorch DataLoader.
        return len(self.data)

    def __getitem__(self, idx):
        file_path = self.data.iloc[idx]['file_path']
        label = self.data.iloc[idx]['label']
        modality = self.data.iloc[idx]['modality']

        scan_data, _ = load_nifti_file(file_path)  #Load the NIfTI file and ignore metadata since only voxel data is required.

        scan_data = scan_data[np.newaxis, :, :, :]  #Explicitly add a channel dimension to match PyTorch CNN input expectations.

        scan_tensor = torch.from_numpy(scan_data.astype(np.float32))  #Convert scan data to a float32 tensor for numerical stability.

        if self.transform:
            scan_tensor = self.transform(scan_tensor)  #Apply optional data transformations like normalisation or augmentation.

        #Return a dictionary for flexible access to image, label, file path, and modality during model training or evaluation.
        return {'image': scan_tensor, 'label': torch.tensor(label, dtype=torch.long), 'path': file_path, 'modality': modality}


In [ ]:
#Create dataloaders for MRI, PET, and co-registered datasets to streamline batch processing and ensure consistent data feeding during training and evaluation.
def createDataloaders(train_df, val_df, test_df, batch_size=4):

    #Create separate datasets per modality to allow independent model training or evaluation on MRI, PET, or co-registered data.
    train_mri_dataset = NIfTIDataset(train_df, modality='mri')
    val_mri_dataset = NIfTIDataset(val_df, modality='mri')
    test_mri_dataset = NIfTIDataset(test_df, modality='mri')

    train_pet_dataset = NIfTIDataset(train_df, modality='pet')
    val_pet_dataset = NIfTIDataset(val_df, modality='pet')
    test_pet_dataset = NIfTIDataset(test_df, modality='pet')

    train_coreg_dataset = NIfTIDataset(train_df, modality='coreg')
    val_coreg_dataset = NIfTIDataset(val_df, modality='coreg')
    test_coreg_dataset = NIfTIDataset(test_df, modality='coreg')

    #Use DataLoaders to efficiently batch, shuffle, and parallel load samples during training; shuffling is disabled for validation and testing to maintain order.
    train_mri_loader = DataLoader(train_mri_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_mri_loader = DataLoader(val_mri_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_mri_loader = DataLoader(test_mri_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    train_pet_loader = DataLoader(train_pet_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_pet_loader = DataLoader(val_pet_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_pet_loader = DataLoader(test_pet_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    train_coreg_loader = DataLoader(train_coreg_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_coreg_loader = DataLoader(val_coreg_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_coreg_loader = DataLoader(test_coreg_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    #Return a structured dictionary for easy modality-specific dataloader access in model training loops.
    return {
        'mri': {'train': train_mri_loader, 'val': val_mri_loader, 'test': test_mri_loader},
        'pet': {'train': train_pet_loader, 'val': val_pet_loader, 'test': test_pet_loader},
        'coreg': {'train': train_coreg_loader, 'val': val_coreg_loader, 'test': test_coreg_loader}
    }

dataloaders = createDataloaders(train_df, val_df, test_df, batch_size=2)

#Quickly verify that dataloaders are correctly created and batch division is consistent across modalities and splits.
print("MRI dataloader sizes:")
print(f"Train: {len(dataloaders['mri']['train'])} batches")
print(f"Val: {len(dataloaders['mri']['val'])} batches")
print(f"Test: {len(dataloaders['mri']['test'])} batches")

print("\nPET dataloader sizes:")
print(f"Train: {len(dataloaders['pet']['train'])} batches")
print(f"Val: {len(dataloaders['pet']['val'])} batches")
print(f"Test: {len(dataloaders['pet']['test'])} batches")

print("\nCo-registered dataloader sizes:")
print(f"Train: {len(dataloaders['coreg']['train'])} batches")
print(f"Val: {len(dataloaders['coreg']['val'])} batches")
print(f"Test: {len(dataloaders['coreg']['test'])} batches")


In [ ]:
#Save the training, validation, and test splits as CSV files to enable reproducibility and future reuse without needing to rerun splits.
train_df.to_csv(os.path.join(BASE_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(BASE_DIR, 'val_split.csv'), index=False)
test_df.to_csv(os.path.join(BASE_DIR, 'test_split.csv'), index=False)

#Provide user confirmation that dataset splits have been successfully saved.
print(f"Saved dataset splits to {BASE_DIR}")

#Save a comprehensive summary to text for traceability and easy inspection of data composition.
with open(os.path.join(BASE_DIR, 'dataset_summary.txt'), 'w') as f:
    f.write(f"Total files: {len(catalogue_df)}\n")
    f.write(f"MRI files: {len(catalogue_df[catalogue_df['modality'] == 'mri'])}\n")
    f.write(f"PET files: {len(catalogue_df[catalogue_df['modality'] == 'pet'])}\n")
    f.write(f"Co-registered files: {len(catalogue_df[catalogue_df['modality'] == 'coreg'])}\n")
    f.write(f"AD samples: {len(catalogue_df[catalogue_df['label'] == 1])}\n")
    f.write(f"CN samples: {len(catalogue_df[catalogue_df['label'] == 0])}\n")
    f.write(f"Original samples: {len(catalogue_df[catalogue_df['is_augmented'] == False])}\n")
    f.write(f"Augmented samples: {len(catalogue_df[catalogue_df['is_augmented'] == True])}\n\n")

    #Include detailed statistics per split to make sure model training datasets are properly balanced and separated.
    f.write("Training set:\n")
    f.write(f"  Total: {len(train_df)} files\n")
    f.write(f"  AD: {len(train_df[train_df['label'] == 1])} files\n")
    f.write(f"  CN: {len(train_df[train_df['label'] == 0])} files\n")
    f.write(f"  MRI: {len(train_df[train_df['modality'] == 'mri'])} files\n")
    f.write(f"  PET: {len(train_df[train_df['modality'] == 'pet'])} files\n")
    f.write(f"  Co-registered: {len(train_df[train_df['modality'] == 'coreg'])} files\n\n")

    f.write("Validation set:\n")
    f.write(f"  Total: {len(val_df)} files\n")
    f.write(f"  AD: {len(val_df[val_df['label'] == 1])} files\n")
    f.write(f"  CN: {len(val_df[val_df['label'] == 0])} files\n")
    f.write(f"  MRI: {len(val_df[val_df['modality'] == 'mri'])} files\n")
    f.write(f"  PET: {len(val_df[val_df['modality'] == 'pet'])} files\n")
    f.write(f"  Co-registered: {len(val_df[val_df['modality'] == 'coreg'])} files\n\n")

    f.write("Test set:\n")
    f.write(f"  Total: {len(test_df)} files\n")
    f.write(f"  AD: {len(test_df[test_df['label'] == 1])} files\n")
    f.write(f"  CN: {len(test_df[test_df['label'] == 0])} files\n")
    f.write(f"  MRI: {len(test_df[test_df['modality'] == 'mri'])} files\n")
    f.write(f"  PET: {len(test_df[test_df['modality'] == 'pet'])} files\n")
    f.write(f"  Co-registered: {len(test_df[test_df['modality'] == 'coreg'])} files\n")

#Confirm to the user that the dataset summary has been successfully created and saved.
print(f"Saved dataset summary to {os.path.join(BASE_DIR, 'dataset_summary.txt')}")


In [ ]:
#Visualise 3D medical images by extracting representative 2D slices for qualitative inspection of MRI, PET, and co-registered scans.
def sampleVisualisation(dataset, idx=0, modality=""):
    sample = dataset[idx]
    image = sample['image']
    label = sample['label']

    #Display basic tensor shape to help diagnose unexpected input dimensions early.
    print(f"Image shape: {image.shape}")

    #Handle standard volumetric formats; channel-first convention is assumed for single-channel data.
    if len(image.shape) == 4 and image.shape[0] == 1:
        volume = image[0]
    #Handle rare formats where channels are last; fallback to first channel.
    elif len(image.shape) == 4 and image.shape[3] == 4:
        volume = image[:, :, :, 0]
    else:
        #Gracefully exit if an unexpected format is encountered to avoid downstream errors.
        print(f"Unexpected image shape: {image.shape}")
        return

    #Extract central slices along each major anatomical plane for informative and consistent visualisation.
    middle_z = volume.shape[0] // 2
    middle_y = volume.shape[1] // 2
    middle_x = volume.shape[2] // 2

    slice_x = volume[middle_z, :, :]
    slice_y = volume[:, middle_y, :]
    slice_z = volume[:, :, middle_x]

    #Plot three orthogonal views to provide spatial context about the brain's internal structure.
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(slice_x, cmap='gray')
    axes[0].set_title(f"Sagittal Slice")

    axes[1].imshow(slice_y, cmap='gray')
    axes[1].set_title(f"Coronal Slice")

    axes[2].imshow(slice_z, cmap='gray')
    axes[2].set_title(f"Axial Slice")

    #Annotate the plots with modality and diagnosis to aid visual comparison across samples.
    fig.suptitle(f"{modality.upper()} - {'AD' if label == 1 else 'CN'}")
    plt.tight_layout()
    plt.show()

#Visually check MRI samples to confirm structural features are correctly captured.
print("Visualising MRI sample:")
sampleVisualisation(dataloaders['mri']['train'].dataset, idx=0, modality="mri")

#Visually check PET samples to verify chemical activity mapping.
print("Visualising PET sample:")
sampleVisualisation(dataloaders['pet']['train'].dataset, idx=0, modality="pet")

#Visua
#Visually check PET samples to verify chemical activity mapping.
print("Visualising PET sample:")
sampleVisualisation(dataloaders['pet']['train'].dataset, idx=0, modality="pet")

#Visually check co-registered samples to ensure combined modalities align correctly.
print("Visualising co-registered sample:")
sampleVisualisation(dataloaders['coreg']['train'].dataset, idx=0, modality="coreg")